# Pydantic to structure Gemini outputs

In [ ]:
# from dotenv import load_dotenv
# import os 
from google import genai

# load_dotenv()

client = genai.Client()

response = client.models.generate_content(
    model = "gemini-2.5-flash", contents="Tell me a programming joke"
)

response



GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text="""Why do programmers always confuse Halloween and Christmas?

Because Oct 31 == Dec 25!"""
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-2.5-flash',
  response_id='aY2-aPjRCPHYvdIPzoSFIA',
  sdk_http_response=HttpResponse(
    headers=<dict len=11>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=21,
    prompt_token_count=6,
    prompt_tokens_details=[
      ModalityTokenCount(
        modality=<MediaModality.TEXT: 'TEXT'>,
        token_count=6
      ),
    ],
    thoughts_token_count=1349,
    total_token_count=1376
  )
)

In [4]:
response.text

'Why do programmers always confuse Halloween and Christmas?\n\nBecause Oct 31 == Dec 25!'

In [5]:
print(response.text)

Why do programmers always confuse Halloween and Christmas?

Because Oct 31 == Dec 25!


In [6]:
def ask_llm(prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

ask_llm("Du är en Göteborgare, ge mig ett skämt som är go")

'Hallå där, min vän! Ska du ha ett skämt som är gött å go, då får du la lyssna på den här, den är ju klassisk men den funkar alltid här i Götet:\n\nDet sitter en riktig göteborgsgubbe på en bänk vid Kungsportsplatsen, det duggar förstås som det brukar.\nDå kommer en turist fram och frågar lite försynt:\n– Ursäkta, men regnar det verkligen alltid så här mycket i Göteborg?\n\nGubben tittar upp, tar ett djupt andetag och säger lugnt:\n– Näää, inte *alltid*. Bara de dagar när jag glömt paraplyet. Å sen alla de dagar när jag har paraplyet med mig, ja då är de ju strålande sol! Men idag, idag glömde jag de jävla paraplyet igen! Jäkla skitväder, men gött ändå la!\n\nDen sitter la fint, eller hur? Hah! Den är ju lite sån där "gött-å-klaga-men-ändå-va-nöjd"-humor som vi gillar här på västkusten!'

## Try to get data from our LLM

In [11]:
response = ask_llm("""
    Du är en expert inom köp och sälj av bostäder, likt en proffsig mäklare.
    Generera bostadspriser, månadsavgifter, address, stad, boarea i jsonformat (ej markdown)

    Exempel:
            {
                "address": "Fågelvägen 5,
                "price_sek": 3000000,
                "city": "Göteborg",
                "monthly_fee": 4000,
                "area": 60
            }   
                   
    Ge mig en lista på 5 bostäder
""")

response

'[\n    {\n        "address": "Birger Jarlsgatan 12, Lgh 401",\n        "price_sek": 7500000,\n        "city": "Stockholm",\n        "monthly_fee": 3850,\n        "area": 75\n    },\n    {\n        "address": "Linnégatan 34B",\n        "price_sek": 4250000,\n        "city": "Göteborg",\n        "monthly_fee": 3200,\n        "area": 65\n    },\n    {\n        "address": "Davidshallsgatan 8, Vån 2",\n        "price_sek": 3100000,\n        "city": "Malmö",\n        "monthly_fee": 2950,\n        "area": 70\n    },\n    {\n        "address": "Östra Ågatan 22, Lgh 3",\n        "price_sek": 2800000,\n        "city": "Uppsala",\n        "monthly_fee": 3100,\n        "area": 55\n    },\n    {\n        "address": "Storgatan 15B",\n        "price_sek": 2200000,\n        "city": "Örebro",\n        "monthly_fee": 3500,\n        "area": 80\n    }\n]'

In [12]:
print(response)

[
    {
        "address": "Birger Jarlsgatan 12, Lgh 401",
        "price_sek": 7500000,
        "city": "Stockholm",
        "monthly_fee": 3850,
        "area": 75
    },
    {
        "address": "Linnégatan 34B",
        "price_sek": 4250000,
        "city": "Göteborg",
        "monthly_fee": 3200,
        "area": 65
    },
    {
        "address": "Davidshallsgatan 8, Vån 2",
        "price_sek": 3100000,
        "city": "Malmö",
        "monthly_fee": 2950,
        "area": 70
    },
    {
        "address": "Östra Ågatan 22, Lgh 3",
        "price_sek": 2800000,
        "city": "Uppsala",
        "monthly_fee": 3100,
        "area": 55
    },
    {
        "address": "Storgatan 15B",
        "price_sek": 2200000,
        "city": "Örebro",
        "monthly_fee": 3500,
        "area": 80
    }
]


## Parse and validate data

In [21]:
from pydantic import BaseModel, Field
import json 

class Apartment(BaseModel):
    address: str 
    city: str 
    price_sek: int = Field(gt=1000000, lt = 8000000) 
    monthly_fee: int 
    area: int 

class ApartmentList(BaseModel):
    objects: list[Apartment]


apartments = ApartmentList.model_validate({"objects": json.loads(response)})
apartments
    

ApartmentList(objects=[Apartment(address='Birger Jarlsgatan 12, Lgh 401', city='Stockholm', price_sek=7500000, monthly_fee=3850, area=75), Apartment(address='Linnégatan 34B', city='Göteborg', price_sek=4250000, monthly_fee=3200, area=65), Apartment(address='Davidshallsgatan 8, Vån 2', city='Malmö', price_sek=3100000, monthly_fee=2950, area=70), Apartment(address='Östra Ågatan 22, Lgh 3', city='Uppsala', price_sek=2800000, monthly_fee=3100, area=55), Apartment(address='Storgatan 15B', city='Örebro', price_sek=2200000, monthly_fee=3500, area=80)])

In [23]:
apartments.objects

[Apartment(address='Birger Jarlsgatan 12, Lgh 401', city='Stockholm', price_sek=7500000, monthly_fee=3850, area=75),
 Apartment(address='Linnégatan 34B', city='Göteborg', price_sek=4250000, monthly_fee=3200, area=65),
 Apartment(address='Davidshallsgatan 8, Vån 2', city='Malmö', price_sek=3100000, monthly_fee=2950, area=70),
 Apartment(address='Östra Ågatan 22, Lgh 3', city='Uppsala', price_sek=2800000, monthly_fee=3100, area=55),
 Apartment(address='Storgatan 15B', city='Örebro', price_sek=2200000, monthly_fee=3500, area=80)]

In [27]:
apartments.objects[1].address, apartments.objects[1].city

('Linnégatan 34B', 'Göteborg')

In [31]:
addresses = [apartment.address for apartment in apartments.objects ]

addresses

['Birger Jarlsgatan 12, Lgh 401',
 'Linnégatan 34B',
 'Davidshallsgatan 8, Vån 2',
 'Östra Ågatan 22, Lgh 3',
 'Storgatan 15B']

In [ ]:
addresses = [
    apartment.address
    for apartment in apartments.objects
    if apartment.price_sek < 4000000
]

addresses

['Davidshallsgatan 8, Vån 2', 'Östra Ågatan 22, Lgh 3', 'Storgatan 15B']

Get address, city, price, monthly_fee for the interval 4M - 8M 

In [40]:
addresses_4_to_8 = [
    [home.address, home.city, home.price_sek, home.monthly_fee]
    for home
    in apartments.objects
    if 4000000 < home.price_sek < 8000000
    # if home.price_sek > 4000000 and home.price_sek < 8000000
]

addresses_4_to_8

[['Birger Jarlsgatan 12, Lgh 401', 'Stockholm', 7500000, 3850],
 ['Linnégatan 34B', 'Göteborg', 4250000, 3200]]

### convert to dataframe

In [44]:
import pandas as pd

# Filter
filtered_homes = [
    home for home in apartments.objects if 4_000_000 < home.price_sek < 8_000_000
]

filtered_homes

[Apartment(address='Birger Jarlsgatan 12, Lgh 401', city='Stockholm', price_sek=7500000, monthly_fee=3850, area=75),
 Apartment(address='Linnégatan 34B', city='Göteborg', price_sek=4250000, monthly_fee=3200, area=65)]

In [48]:
# Convert to df
df_homes_filtered = pd.DataFrame(
    [
        home.model_dump(include={"address", "city", "price_sek", "monthly_fee"})
        for home in filtered_homes
    ]
)

df_homes_filtered

,address,city,price_sek,monthly_fee
0,"Birger Jarlsgatan 12, Lgh 401",Stockholm,7500000,3850
1,Linnégatan 34B,Göteborg,4250000,3200


In [49]:
df_homes_filtered.to_csv("filtered_homes.csv", index=False)

## save to json - serialize pydantic model

In [51]:
apartments

ApartmentList(objects=[Apartment(address='Birger Jarlsgatan 12, Lgh 401', city='Stockholm', price_sek=7500000, monthly_fee=3850, area=75), Apartment(address='Linnégatan 34B', city='Göteborg', price_sek=4250000, monthly_fee=3200, area=65), Apartment(address='Davidshallsgatan 8, Vån 2', city='Malmö', price_sek=3100000, monthly_fee=2950, area=70), Apartment(address='Östra Ågatan 22, Lgh 3', city='Uppsala', price_sek=2800000, monthly_fee=3100, area=55), Apartment(address='Storgatan 15B', city='Örebro', price_sek=2200000, monthly_fee=3500, area=80)])

In [ ]:
# dictionary
apartments.model_dump()

{'objects': [{'address': 'Birger Jarlsgatan 12, Lgh 401',
   'city': 'Stockholm',
   'price_sek': 7500000,
   'monthly_fee': 3850,
   'area': 75},
  {'address': 'Linnégatan 34B',
   'city': 'Göteborg',
   'price_sek': 4250000,
   'monthly_fee': 3200,
   'area': 65},
  {'address': 'Davidshallsgatan 8, Vån 2',
   'city': 'Malmö',
   'price_sek': 3100000,
   'monthly_fee': 2950,
   'area': 70},
  {'address': 'Östra Ågatan 22, Lgh 3',
   'city': 'Uppsala',
   'price_sek': 2800000,
   'monthly_fee': 3100,
   'area': 55},
  {'address': 'Storgatan 15B',
   'city': 'Örebro',
   'price_sek': 2200000,
   'monthly_fee': 3500,
   'area': 80}]}

In [54]:
# str of json data
apartments.model_dump_json()

'{"objects":[{"address":"Birger Jarlsgatan 12, Lgh 401","city":"Stockholm","price_sek":7500000,"monthly_fee":3850,"area":75},{"address":"Linnégatan 34B","city":"Göteborg","price_sek":4250000,"monthly_fee":3200,"area":65},{"address":"Davidshallsgatan 8, Vån 2","city":"Malmö","price_sek":3100000,"monthly_fee":2950,"area":70},{"address":"Östra Ågatan 22, Lgh 3","city":"Uppsala","price_sek":2800000,"monthly_fee":3100,"area":55},{"address":"Storgatan 15B","city":"Örebro","price_sek":2200000,"monthly_fee":3500,"area":80}]}'

In [56]:
with open("apartments.json", "w") as json_file:
    json_file.write(apartments.model_dump_json(indent=3))